# Path A, P11 — the 2x2 (Oct 27, 2026)

Pre-registered in the Path A Workbook, Ch. A8.2 / A8.4. No choice below was changed.

- **Data:** `data/processed/hazard_table.parquet` (P8), all training cells (Hierarchy, Full, Star), all rows (bot runs and incomplete runs kept, as in the table). G-train rows fit the models; G-val rows score them. lab = 1 if dataset is EL or CL, 0 if CC.
- **Models** (R0 structure: const + lab + features; unpenalized logistic, statsmodels Logit, raw features; `fit.fit`), outcome `y_rev`:
  - M1 = A, |s| (`abs_s`), `at_end`
  - M2 = M1 + trend (w = 1 s)
  - M2' = M1 + e (`time_at_end_log1p`), r (`time_since_rev_log1p`)
  - M3 = all six
- **Comparisons:** M2 − M1, M2' − M1, M3 − M2', M3 − M2. delta = sum over G-val bins of (ln L_big − ln L_small) / n bins; 95% CI from 10,000 resamples of G-val groups (log files) with replacement, seed 573; betas fixed, no refitting (`fit.boot_delta`). Label: "CI above 0" or "CI includes 0".
- **Secondary, descriptive only:** the same 4 models with outcome `y_probe` (bins where a reversal starts stay in, with `y_probe` as in the table: 0 unless a probe also starts in the bin). Checks 1, 2, 5; M3 betas; the 4 deltas.

Functions: `fit.fit`, `fit.bin_ll`, `fit.boot_delta` (`src/slider/fit.py`). Saved: `results/tables/p11_2x2.csv`, `p11_coefs.csv`, `p11_probe.csv`.

In [1]:
import warnings

import numpy as np
import pandas as pd
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning

from slider import config
from slider import fit

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

ht = pd.read_parquet(config.HAZARD_TABLE)
ht = ht[ht["topology"].isin(config.TRAIN_TOPOLOGIES)]
train, val = ht[ht["split"] == "train"], ht[ht["split"] == "val"]
lab_train = train["dataset"].isin(["EL", "CL"]).to_numpy()

print("topologies:", sorted(ht["topology"].unique()))
ht.assign(both=(ht["y_rev"] == 1) & (ht["y_probe"] == 1)).groupby("split").agg(
    rows=("y_rev", "size"), sum_y_rev=("y_rev", "sum"), sum_y_probe=("y_probe", "sum"),
    bins_with_y_rev_and_y_probe=("both", "sum"), groups=("group_id", "nunique"),
    rows_bot_run=("bot_run", "sum"), rows_incomplete=("incomplete", "sum"))

topologies: ['Full', 'Hierarchy', 'Star']


,rows,sum_y_rev,sum_y_probe,bins_with_y_rev_and_y_probe,groups,rows_bot_run,rows_incomplete
split,,,,,,,
train,336073,6073,1491,1,39,37188,3116
val,225033,3862,828,0,21,39475,0


## Fits on G-train and checks

In [2]:
M1 = ["A", "abs_s", "at_end"]
MODELS = {"M1": M1, "M2": M1 + ["trend"], "M2'": M1 + ["time_at_end_log1p", "time_since_rev_log1p"], "M3": fit.FEATURES}
COMPARISONS = [("M2", "M1"), ("M2'", "M1"), ("M3", "M2'"), ("M3", "M2")]
P8_TRAIN = {"rows": 336073, "y_rev": 6073, "y_probe": 1491}    # G-train, all cells: p8_hazard_table.ipynb, by split


def fit_models(outcome):
    """Fit the four models of outcome on G-train; keep every warning each fit raises."""
    out = {}
    for name, feats in MODELS.items():
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            out[name] = (fit.fit(train, feats, False, outcome=outcome), caught)
    return out


def checks_125(models, outcome):
    """Checks 1 (fit health), 2 (sum h_hat = sum y: overall, lab, cloud) and 5 (rows, sum y) for each model."""
    rows, y = [], train[outcome].to_numpy()
    for name, (m, w) in models.items():
        sep = [x for x in w if issubclass(x.category, PerfectSeparationWarning)]
        rows.append({"outcome": outcome, "model": name, "check": 1, "what": "converged, betas finite, no separation warning",
                     "numbers": f"converged={m.mle_retvals['converged']}, iterations={m.mle_retvals['iterations']}, "
                                f"finite={np.isfinite(m.params).all()}, separation warnings={len(sep)}, all warnings={len(w)}"
                                + "".join(f" | {x.category.__name__}: {x.message}" for x in w),
                     "pass": bool(m.mle_retvals["converged"] and np.isfinite(m.params).all() and not sep)})
        h = np.asarray(m.predict())
        for part, mask in [("overall", np.ones(len(train), bool)), ("lab", lab_train), ("cloud", ~lab_train)]:
            diff = h[mask].sum() - y[mask].sum()
            rows.append({"outcome": outcome, "model": name, "check": 2, "what": f"sum(h_hat) == sum({outcome}), {part}",
                         "numbers": f"sum h = {h[mask].sum():.6f}, sum y = {y[mask].sum()}, diff = {diff:.2e}",
                         "pass": bool(abs(diff) < 1e-3)})
        n, s = int(m.nobs), int(m.model.endog.sum())
        rows.append({"outcome": outcome, "model": name, "check": 5, "what": f"G-train rows and sum({outcome}) = table subset = P8",
                     "numbers": f"fit: {n}, {s} | table: {len(train)}, {int(train[outcome].sum())} | P8: {P8_TRAIN['rows']}, {P8_TRAIN[outcome]}",
                     "pass": n == len(train) == P8_TRAIN["rows"] and s == train[outcome].sum() == P8_TRAIN[outcome]})
    return rows


def deltas(models):
    """delta (big - small) per G-val bin and 95% CI for each comparison."""
    ll = {name: fit.bin_ll(m, val) for name, (m, w) in models.items()}
    rows = []
    for big, small in COMPARISONS:
        delta, (low, high) = fit.boot_delta(ll[big], ll[small], val["group_id"])
        rows.append({"comparison": f"{big} - {small}", "delta": delta, "ci_low": low, "ci_high": high})
    return pd.DataFrame(rows)


def coef_table(models):
    return pd.concat([pd.DataFrame({"model": name, "term": m.params.index, "beta": m.params.to_numpy(), "se": m.bse.to_numpy()})
                      for name, (m, w) in models.items()], ignore_index=True)


rev = fit_models("y_rev")
checks = checks_125(rev, "y_rev")
for big, small in [("M2", "M1"), ("M2'", "M1"), ("M3", "M2"), ("M3", "M2'")]:      # 3. G-train LL order
    a, b = rev[big][0].llf, rev[small][0].llf
    checks.append({"outcome": "y_rev", "model": f"{big} vs {small}", "check": 3, "what": f"G-train LL: {big} >= {small}",
                   "numbers": f"LL {big} = {a:.4f}, LL {small} = {b:.4f}, diff = {a - b:.4f}", "pass": bool(a >= b)})
for name, (m, w) in rev.items():                                                    # 4. signs and size
    bA = m.params["A"]
    checks.append({"outcome": "y_rev", "model": name, "check": 4, "what": "beta_A < 0", "numbers": f"beta_A = {bA:.4f}", "pass": bool(bA < 0)})
    checks.append({"outcome": "y_rev", "model": name, "check": 4, "what": "beta_A roughly -1 to -4", "numbers": f"beta_A = {bA:.4f}",
                   "pass": bool(-4 <= bA <= -1)})
    if "trend" in m.params:
        bt = m.params["trend"]
        checks.append({"outcome": "y_rev", "model": name, "check": 4, "what": "beta_trend < 0", "numbers": f"beta_trend = {bt:.4f}",
                       "pass": bool(bt < 0)})
checks = pd.DataFrame(checks).sort_values(["check", "model"], kind="stable", ignore_index=True)
with pd.option_context("display.max_colwidth", 200):
    display(checks)
print("all checks pass:", bool(checks["pass"].all()))

,outcome,model,check,what,numbers,pass
0,y_rev,M1,1,"converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
1,y_rev,M2,1,"converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
2,y_rev,M2',1,"converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
3,y_rev,M3,1,"converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
4,y_rev,M1,2,"sum(h_hat) == sum(y_rev), overall","sum h = 6073.000000, sum y = 6073, diff = -3.64e-12",True
5,y_rev,M1,2,"sum(h_hat) == sum(y_rev), lab","sum h = 3238.000000, sum y = 3238, diff = -2.27e-12",True
6,y_rev,M1,2,"sum(h_hat) == sum(y_rev), cloud","sum h = 2835.000000, sum y = 2835, diff = -2.27e-12",True
7,y_rev,M2,2,"sum(h_hat) == sum(y_rev), overall","sum h = 6073.000000, sum y = 6073, diff = -9.09e-13",True
8,y_rev,M2,2,"sum(h_hat) == sum(y_rev), lab","sum h = 3238.000000, sum y = 3238, diff = -9.09e-13",True
9,y_rev,M2,2,"sum(h_hat) == sum(y_rev), cloud","sum h = 2835.000000, sum y = 2835, diff = -4.55e-13",True


all checks pass: True


## The 2x2: delta on G-val, per bin

In [3]:
def ci_label(low, high):
    return "CI above 0" if low > 0 else "CI includes 0" if high >= 0 else "CI below 0"


two = deltas(rev)
two["ci"] = [ci_label(lo, hi) for lo, hi in zip(two["ci_low"], two["ci_high"])]
two["n_val_bins"], two["n_val_groups"] = len(val), val["group_id"].nunique()
two.to_csv(config.P11_2X2, index=False)
with pd.option_context("display.float_format", "{:.3e}".format):
    display(two)

,comparison,delta,ci_low,ci_high,ci,n_val_bins,n_val_groups
0,M2 - M1,2.091e-04,-1.280e-04,4.743e-04,CI includes 0,225033,21
1,M2' - M1,2.862e-03,1.531e-03,4.748e-03,CI above 0,225033,21
2,M3 - M2',4.671e-04,-1.213e-05,8.276e-04,CI includes 0,225033,21
3,M3 - M2,3.120e-03,1.816e-03,4.901e-03,CI above 0,225033,21


## Weights with SEs, all four models (outcome y_rev)

In [4]:
coefs = coef_table(rev)
coefs.to_csv(config.P11_COEFS, index=False)
TERMS = ["const", "lab"] + fit.FEATURES
wide = coefs.pivot(index="term", columns="model", values=["beta", "se"]).swaplevel(axis=1)
wide[[(m, v) for m in MODELS for v in ["beta", "se"]]].reindex(TERMS).round(4)

model                     M1              M2             M2'              M3        
                        beta      se    beta      se    beta      se    beta      se
term                                                                                
const                -3.2697  0.0481 -3.3216  0.0483 -3.1941  0.0538 -3.2779  0.0544
lab                   0.0052  0.0265  0.0243  0.0267  0.0578  0.0271  0.0919  0.0273
A                    -2.5800  0.0450 -2.3754  0.0467 -2.1074  0.0477 -1.7359  0.0505
trend                    NaN     NaN -1.2385  0.0811     NaN     NaN -1.6799  0.0785
abs_s                 0.9629  0.0884  0.9161  0.0885  1.0476  0.0879  0.9686  0.0879
at_end               -0.3653  0.0572 -0.4005  0.0573  0.4446  0.0615  0.4997  0.0617
time_at_end_log1p        NaN     NaN     NaN     NaN -0.3774  0.0241 -0.4247  0.0243
time_since_rev_log1p     NaN     NaN     NaN     NaN -0.2093  0.0248 -0.2199  0.0248

## Secondary (descriptive only): outcome y_probe

Same four models and comparisons, outcome `y_probe`. Checks 1, 2 (sum(h_hat) == sum(y_probe)) and 5.

In [5]:
probe = fit_models("y_probe")
probe_checks = pd.DataFrame(checks_125(probe, "y_probe")).sort_values(["check", "model"], kind="stable", ignore_index=True)
with pd.option_context("display.max_colwidth", 200):
    display(probe_checks)
print("all probe checks pass:", bool(probe_checks["pass"].all()))

,outcome,model,check,what,numbers,pass
0,y_probe,M1,1,"converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
1,y_probe,M2,1,"converged, betas finite, no separation warning","converged=True, iterations=9, finite=True, separation warnings=0, all warnings=0",True
2,y_probe,M2',1,"converged, betas finite, no separation warning","converged=True, iterations=10, finite=True, separation warnings=0, all warnings=0",True
3,y_probe,M3,1,"converged, betas finite, no separation warning","converged=True, iterations=10, finite=True, separation warnings=0, all warnings=0",True
4,y_probe,M1,2,"sum(h_hat) == sum(y_probe), overall","sum h = 1491.000000, sum y = 1491, diff = -6.82e-13",True
5,y_probe,M1,2,"sum(h_hat) == sum(y_probe), lab","sum h = 805.000000, sum y = 805, diff = -6.82e-13",True
6,y_probe,M1,2,"sum(h_hat) == sum(y_probe), cloud","sum h = 686.000000, sum y = 686, diff = -1.14e-13",True
7,y_probe,M2,2,"sum(h_hat) == sum(y_probe), overall","sum h = 1491.000000, sum y = 1491, diff = 9.09e-13",True
8,y_probe,M2,2,"sum(h_hat) == sum(y_probe), lab","sum h = 805.000000, sum y = 805, diff = 1.25e-12",True
9,y_probe,M2,2,"sum(h_hat) == sum(y_probe), cloud","sum h = 686.000000, sum y = 686, diff = -2.27e-13",True


all probe checks pass: True


In [6]:
pdel = deltas(probe)
m3 = coef_table({"M3": probe["M3"]})
probe_out = pd.concat([
    pd.DataFrame({"part": "delta", "name": pdel["comparison"], "estimate": pdel["delta"], "se": np.nan,
                  "ci_low": pdel["ci_low"], "ci_high": pdel["ci_high"]}),
    pd.DataFrame({"part": "M3 beta", "name": m3["term"], "estimate": m3["beta"], "se": m3["se"], "ci_low": np.nan, "ci_high": np.nan}),
], ignore_index=True).assign(label="descriptive")
probe_out.to_csv(config.P11_PROBE, index=False)
with pd.option_context("display.float_format", "{:.4g}".format):
    display(probe_out)

,part,name,estimate,se,ci_low,ci_high,label
0,delta,M2 - M1,5.309e-06,NaN,-2.485e-06,1.332e-05,descriptive
1,delta,M2' - M1,0.0001708,NaN,2.453e-05,0.0002967,descriptive
2,delta,M3 - M2',2.751e-05,NaN,9.443e-06,4.396e-05,descriptive
3,delta,M3 - M2,0.000193,NaN,4.177e-05,0.0003224,descriptive
4,M3 beta,const,-5.31,0.1223,NaN,NaN,descriptive
5,M3 beta,lab,0.0738,0.05337,NaN,NaN,descriptive
6,M3 beta,A,0.1544,0.09538,NaN,NaN,descriptive
7,M3 beta,trend,-0.6489,0.1933,NaN,NaN,descriptive
8,M3 beta,abs_s,0.201,0.2063,NaN,NaN,descriptive
9,M3 beta,at_end,0.2986,0.1494,NaN,NaN,descriptive
